# Chapter 3: Group By, Having & Count

### Core Concepts:
* **`COUNT()`**: An aggregate function that counts rows.
* **`GROUP BY`**: Groups rows that share the same values in specified columns.
* **`HAVING`**: Filters groups *after* they have been aggregated (unlike `WHERE`, which filters individual rows *before* aggregation).

In this lesson, we will query the Chicago Crime dataset to find the most common types of crimes.

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client()

## Task 1: View Table Schema

Let's remind ourselves of the `chicago_crime.crime` table schema to see which columns we want to group and aggregate.

In [ ]:
dataset_ref = client.dataset("chicago_crime", project="bigquery-public-data")
table_ref = dataset_ref.table("crime")
table = client.get_table(table_ref)

for field in table.schema:
    if field.name in ["primary_type", "id", "year", "description"]:
        print(f"{field.name:<15} | {field.field_type:<10} | {field.description}")

## Task 2: Write the Grouping Query

Open the file `queries/crime_types.sql`.

Write a query that:
1. Selects the `primary_type` column.
2. Counts the number of incidents using `COUNT(id)` and aliases it as `num_incidents`.
3. Groups by the `primary_type` column.
4. Filters the groups using `HAVING` to only include crime types with more than 200,000 total incidents.

Once you have written it, run the block below to load it.

In [ ]:
# Load query from local SQL file
with open("queries/crime_types.sql", "r") as file:
    query_crimes = file.read()

print("Query Loaded:")
print(query_crimes)

## Task 3: Dry Run

Check how much data this query scans before executing it.

In [ ]:
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = client.query(query_crimes, job_config=dry_run_config)

print(f"This query will scan: {dry_run_job.total_bytes_processed / (1024**2):.2f} MB of data.")

## Task 4: Execute

Run the query with a safety limit of 100 MB.

In [ ]:
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=100 * 1024 * 1024)

try:
    query_job = client.query(query_crimes, job_config=safe_config)
    df_crimes = query_job.to_dataframe()
    print(f"Query returned {len(df_crimes)} rows.")
    print(df_crimes)
except Exception as e:
    print(f"Query failed/blocked: {e}")